<a href="https://colab.research.google.com/github/maikol0629/dl_voice_command_cl/blob/main/06_evaluacion_robustez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación de Robustez: Comparación de Modelos frente a Ataques Adversariales

Este notebook carga los tres modelos entrenados (CNN Baseline, CRNN y SpectroTransformer) y los evalúa tanto en el conjunto de validación limpio como en el conjunto de test adversarial. Se comparan las métricas de accuracy y se analiza la caída de rendimiento frente a perturbaciones adversarias.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install numpy pandas matplotlib scikit-learn librosa soundfile torch torchaudio
    %cd /content/drive/MyDrive/dl_voice_command_cl

import os
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio.transforms as T

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
SAMPLE_RATE = 16000
N_CLASSES = 35
BATCH_SIZE = 64
N_MELS = 64
N_FFT = 1024
HOP_LENGTH = 256
MAX_SAMPLES = SAMPLE_RATE
SEED = 42

DATA_PATH = Path('./data')
TRAIN_AUDIO_DIR = DATA_PATH / 'train/train/train'
ADV_TEST_DIR = DATA_PATH / 'adv_test'

np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
class VoiceCommandsDataset(Dataset):
    def __init__(self, df, audio_dir, max_samples=MAX_SAMPLES):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.max_samples = max_samples

    def __len__(self):
        return len(self.df)

    def _normalize_length(self, audio):
        if audio.shape[0] > self.max_samples:
            return audio[:self.max_samples]
        if audio.shape[0] < self.max_samples:
            return np.pad(audio, (0, self.max_samples - audio.shape[0]), mode='constant')
        return audio

    def _load_audio(self, file_name):
        path = self.audio_dir / file_name
        arr = np.load(path, allow_pickle=False)
        if isinstance(arr, np.lib.npyio.NpzFile):
            key = 'audio' if 'audio' in arr.files else arr.files[0]
            arr = arr[key]
        return np.asarray(arr).reshape(-1).astype(np.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio = self._load_audio(row['filename'])
        audio = self._normalize_length(audio)
        label = int(row['label_id'])
        return torch.tensor(audio, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

train_meta = pd.read_csv('train_metadata.csv')
if 'label' not in train_meta.columns or train_meta['label'].isna().any():
    meta_df = pd.read_csv(DATA_PATH / 'train' / 'metadata.csv')
    meta_df = meta_df.rename(columns={'file_name': 'filename'})
    if 'filename' not in train_meta.columns:
        train_meta['filename'] = train_meta['file_path'].apply(lambda p: Path(p).name)
    train_meta = train_meta.drop(columns=['label'], errors='ignore').merge(meta_df, on='filename', how='left')

train_meta = train_meta.dropna(subset=['label']).copy()
train_meta['label'] = train_meta['label'].astype(str)

label_encoder = LabelEncoder()
train_meta['label_id'] = label_encoder.fit_transform(train_meta['label'])
classes = label_encoder.classes_
print(f"Clases: {len(classes)}")

_, val_df = train_test_split(
    train_meta, test_size=0.2, random_state=SEED, stratify=train_meta['label_id']
)

val_dataset = VoiceCommandsDataset(val_df, TRAIN_AUDIO_DIR)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Val muestras: {len(val_dataset)}")

adv_meta_path = ADV_TEST_DIR / 'metadata.csv'
if adv_meta_path.exists():
    adv_meta = pd.read_csv(adv_meta_path)
    if 'label' in adv_meta.columns:
        adv_meta['label'] = adv_meta['label'].astype(str)
        adv_meta['label_id'] = label_encoder.transform(adv_meta['label'])
        adv_dataset = VoiceCommandsDataset(adv_meta, ADV_TEST_DIR)
        adv_loader = DataLoader(adv_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        print(f"Adv test muestras: {len(adv_dataset)}")
    else:
        print("adv_test metadata.csv no contiene columna 'label'")
        adv_loader = None
else:
    print("No se encontro metadata.csv para adv_test")
    adv_loader = None

In [ ]:
class AudioToMelSpectrogram(nn.Module):
    def __init__(self, sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS):
        super().__init__()
        self.mel_spec = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        mel = self.mel_spec(x)
        mel_db = self.amplitude_to_db(mel)
        mean = mel_db.mean(dim=[-2, -1], keepdim=True)
        std = mel_db.std(dim=[-2, -1], keepdim=True)
        return (mel_db - mean) / (std + 1e-8)

audio_transform = AudioToMelSpectrogram().to(device)


class BaselineCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.features(x)
        return self.classifier(x)


class CRNNModel(nn.Module):
    def __init__(self, num_classes, n_mels=N_MELS):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Dropout2d(0.2),
        )
        lstm_dim = 128 * (n_mels // 8)
        self.lstm = nn.LSTM(
            lstm_dim, 256, num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.3
        )
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.conv(x)
        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, C * F)
        x, _ = self.lstm(x)
        x = x.mean(dim=1)
        return self.classifier(x)


class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, dim, max_len=128):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq)

    def forward(self, x):
        seq_len = x.shape[1]
        t = torch.arange(seq_len, device=x.device).type_as(self.inv_freq)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos()[None, None, :, :]
        sin = emb.sin()[None, None, :, :]
        return cos, sin


def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_emb(x, cos, sin):
    return x * cos + rotate_half(x) * sin


class RoPESelfAttention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.head_dim = dim // heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.rope = RotaryPositionalEmbedding(self.head_dim)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rope(q)
        q = apply_rotary_emb(q, cos, sin)
        k = apply_rotary_emb(k, cos, sin)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, T, D)
        return self.proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = RoPESelfAttention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mlp_ratio, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class SpectrogramTransformer(nn.Module):
    def __init__(self, num_classes, n_mels=N_MELS, dim=256, depth=6, heads=8):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(n_mels, dim),
            nn.LayerNorm(dim)
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.drop = nn.Dropout(0.1)
        self.blocks = nn.ModuleList([
            TransformerBlock(dim, heads) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.classifier = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x):
        B = x.shape[0]
        x = x.squeeze(1)
        x = x.permute(0, 2, 1)
        x = self.proj(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x[:, 0])
        return self.classifier(x)

In [ ]:
def create_dummy_model(model_class, num_classes, **kwargs):
    model = model_class(num_classes, **kwargs)
    for p in model.parameters():
        nn.init.normal_(p, mean=0.0, std=0.02)
    return model

models_info = []

try:
    cnn = BaselineCNN(N_CLASSES).to(device)
    cnn.load_state_dict(torch.load('best_cnn_baseline.pth', map_location=device))
    cnn.eval()
    models_info.append(('CNN Baseline', cnn))
    print("CNN Baseline: pesos cargados")
except Exception as e:
    print(f"CNN Baseline: no se pudo cargar ({e}), creando dummy")
    cnn = create_dummy_model(BaselineCNN, N_CLASSES).to(device)
    cnn.eval()
    models_info.append(('CNN Baseline', cnn))

try:
    crnn = CRNNModel(N_CLASSES).to(device)
    crnn.load_state_dict(torch.load('best_crnn.pth', map_location=device))
    crnn.eval()
    models_info.append(('CRNN', crnn))
    print("CRNN: pesos cargados")
except Exception as e:
    print(f"CRNN: no se pudo cargar ({e}), creando dummy")
    crnn = create_dummy_model(CRNNModel, N_CLASSES).to(device)
    crnn.eval()
    models_info.append(('CRNN', crnn))

try:
    trans = SpectrogramTransformer(N_CLASSES).to(device)
    trans.load_state_dict(torch.load('best_transformer.pth', map_location=device))
    trans.eval()
    models_info.append(('Transformer', trans))
    print("Transformer: pesos cargados")
except Exception as e:
    print(f"Transformer: no se pudo cargar ({e}), creando dummy")
    trans = create_dummy_model(SpectrogramTransformer, N_CLASSES).to(device)
    trans.eval()
    models_info.append(('Transformer', trans))

In [ ]:
@torch.inference_mode()
def evaluate_model(model, loader, device, model_name):
    model.eval()
    y_true = []
    y_pred = []
    for audios, labels in loader:
        audios = audios.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        mels = audio_transform(audios)
        outputs = model(mels)
        preds = outputs.argmax(dim=1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())
    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true, y_pred, target_names=classes, zero_division=0, digits=4
    )
    return accuracy, cm, report

In [ ]:
print("=" * 60)
print("EVALUACION EN VALIDACION LIMPIA")
print("=" * 60)

val_results = {}
for name, model in models_info:
    acc, cm, report = evaluate_model(model, val_loader, device, name)
    val_results[name] = {'accuracy': acc, 'confusion_matrix': cm, 'report': report}
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(report)

print("\n" + "=" * 60)

In [ ]:
if adv_loader is not None:
    print("=" * 60)
    print("EVALUACION EN TEST ADVERSARIAL")
    print("=" * 60)
else:
    print("=" * 60)
    print("TEST ADVERSARIAL NO DISPONIBLE (sin labels)")
    print("=" * 60)

adv_results = {}
if adv_loader is not None:
    for name, model in models_info:
        acc, cm, report = evaluate_model(model, adv_loader, device, name)
        adv_results[name] = {'accuracy': acc, 'confusion_matrix': cm, 'report': report}
        print(f"\n--- {name} ---")
        print(f"Accuracy: {acc * 100:.2f}%")
        print(report)
else:
    for name, model in models_info:
        adv_results[name] = {'accuracy': 0.0, 'confusion_matrix': None, 'report': ''}
    print("\nNo se pudio evaluar el conjunto adversarial.")

print("\n" + "=" * 60)

In [ ]:
acc_cnn = val_results['CNN Baseline']['accuracy']
acc_crnn = val_results['CRNN']['accuracy']
acc_trans = val_results['Transformer']['accuracy']

adv_cnn = adv_results['CNN Baseline']['accuracy']
adv_crnn = adv_results['CRNN']['accuracy']
adv_trans = adv_results['Transformer']['accuracy']

drop_cnn = (acc_cnn - adv_cnn) * 100
drop_crnn = (acc_crnn - adv_crnn) * 100
drop_trans = (acc_trans - adv_trans) * 100

results = pd.DataFrame({
    'Modelo': ['CNN Baseline', 'CRNN', 'Transformer'],
    'Clean Val Acc': [f'{acc_cnn*100:.2f}%', f'{acc_crnn*100:.2f}%', f'{acc_trans*100:.2f}%'],
    'Adv Test Acc': [f'{adv_cnn*100:.2f}%', f'{adv_crnn*100:.2f}%', f'{adv_trans*100:.2f}%'],
    'Caida (%)': [f'{drop_cnn:.2f}%', f'{drop_crnn:.2f}%', f'{drop_trans:.2f}%']
})

try:
    from IPython.display import display, Markdown
    display(Markdown(results.to_markdown()))
except ImportError:
    print(results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
model_names = ['CNN Baseline', 'CRNN', 'Transformer']
val_accs = [acc_cnn * 100, acc_crnn * 100, acc_trans * 100]
adv_accs_val = [adv_cnn * 100, adv_crnn * 100, adv_trans * 100]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, val_accs, width, label='Clean Validation', color='#2ecc71', edgecolor='black')
bars2 = ax.bar(x + width/2, adv_accs_val, width, label='Adversarial Test', color='#e74c3c', edgecolor='black')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Accuracy (%)')
ax.set_title('Comparacion de Robustez: Clean vs Adversarial')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
if adv_loader is not None:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    for ax, (name, _) in zip(axes, models_info):
        cm = adv_results[name]['confusion_matrix']
        if cm is not None:
            cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True).clip(min=1)
            im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
            ax.set_title(f'{name} - Adv Test')
            ax.set_xlabel('Predicho')
            ax.set_ylabel('Real')
            fig.colorbar(im, ax=ax, fraction=0.046)
    plt.suptitle('Matrices de Confusion Normalizadas - Test Adversarial', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Matrices de confusion no disponibles (sin labels adversariales)")

## Discusión

### ¿Qué modelo fue más robusto?

El **Transformer con RoPE** suele ser el más robusto frente a ataques adversariales debido a su capacidad de atender a patrones globales en el espectrograma, lo que lo hace menos sensible a perturbaciones localizadas. La CRNN ocupa un punto intermedio: las capas convolucionales extraen características locales robustas, y la BiLSTM modela dependencias temporales, pero puede verse afectada si el ataque distorsiona sistemáticamente ciertos marcos temporales. La CNN Baseline, al depender exclusivamente de filtros convolucionales locales sin modelado secuencial explícito, tiende a ser la más vulnerable.

### ¿Por qué una arquitectura maneja mejor las perturbaciones adversariales?

1. **Mecanismo de atención global**: El Transformer con RoPE puede establecer dependencias entre cualquier par de posiciones tiempo-frecuencia, lo que diluye el efecto de perturbaciones localizadas. Al rotar los embeddings posicionales en lugar de sumarlos, RoPE preserva la estructura geométrica del espacio temporal.
2. **Modelado secuencial en CRNN**: La BiLSTM captura la evolución temporal de los fonemas, lo que permite al modelo "corregir" interpretaciones erróneas en un marco temporal usando contexto adyacente.
3. **Sobrerrepresentación de patrones locales en CNN**: La CNN aprende parches locales altamente específicos. Un ataque adversarial que modifica un parche puede engañar fácilmente al clasificador.

### Limitaciones y trabajo futuro

- **Limitaciones**:
  - La evaluación adversarial se realizó sobre un conjunto de test pre-generado; no se generaron ataques ad-hoc (FGSM, PGD) para cada modelo.
  - Los modelos CRNN y Transformer se inicializaron con pesos aleatorios si no se encontraron los archivos `.pth`, por lo que las métricas reportadas pueden no reflejar el rendimiento real entrenado.
  - No se aplicaron técnicas de defensa adversarial (entrenamiento adversarial, suavizado aleatorio, etc.).

- **Trabajo futuro**:
  - Entrenar los modelos CRNN y Transformer desde cero con los mismos datos de entrenamiento.
  - Aplicar ataques adversariales de forma controlada (FGSM, PGD, Carlini-Wagner) para evaluar la robustez de forma más granular.
  - Implementar defensas como entrenamiento adversarial, destilación defensiva, o preprocesamiento de señal.
  - Explorar arquitecturas hybridas (CNN + Transformer) que combinen las ventajas de ambos paradigmas.